# Práctica Final: Comparación de Modelos

## Contexto del Problema
En el sector hotelero, las cancelaciones de reservas generan pérdidas financieras directas e ineficiencias operativas. Este proyecto busca predecir si una reserva va a ser cancelada (`is_canceled = 1`) o no (`is_canceled = 0`) para permitir al hotel tomar medidas preventivas (como campañas de fidelización o gestión de sobreventa controlada).

## Objetivo
Comparar el rendimiento de un modelo lineal (**Regresión Logística**) frente a un modelo basado en reglas de decisión (**Árbol de Decisión**) utilizando un pipeline de preprocesamiento robusto y libre de fuga de datos (*Data Leakage*).

## Justificación de la Métrica Principal (F1-Score)
En el conjunto de datos, las cancelaciones representan aproximadamente el 37% de los casos. Debido a este desbalanceo de clases, usar el **Accuracy** (exactitud) es engañoso: un modelo ingenuo que dijera "nadie cancela" obtendría un 63% de aciertos pero sería inútil. 

Por ello, tomamos el **F1-Score** como métrica principal, ya que busca la **media armónica entre Precisión** (evitar falsas alarmas) y **Recall** (capturar la mayor cantidad de cancelaciones reales).

In [ ]:
# 1. IMPORTACIÓN DE LIBRERÍAS
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# Scikit-learn: Preprocesamiento y Pipelines
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

# Scikit-learn: Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Scikit-learn: Métricas y Evaluación
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

## 2. Carga de Datos y Eliminación de Fuga de Información (*Data Leakage*)

Un error crítico en Machine Learning es incluir variables que "vienen del futuro", es decir, información que solo se conoce **después** de que el evento (la cancelación) ha ocurrido.

* **`reservation_status`**: Indica si la reserva fue completada, cancelada o "no-show". Si dejamos esta columna, el modelo sabrá la respuesta con un 100% de certeza, creando un falso rendimiento perfecto en entrenamiento que fallará en producción.
* **`reservation_status_date`**: La fecha en que cambió el estado de la reserva.

In [ ]:
# Definición de ruta dinámica relativa al proyecto
BASE_DIR = Path(__file__).resolve().parent.parent if "__file__" in locals() else Path.cwd()
DATA_PATH = BASE_DIR / "data" / "raw" / "dataset_practica_final.csv"

# Carga del dataset
df = pd.read_csv(DATA_PATH)

# Definición de target y columnas con fuga de información
target = "is_canceled"
drop_cols = [target, "reservation_status", "reservation_status_date"]

# Separación de características (X) y variable objetivo (y)
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df[target]

# Separación automática de columnas numéricas y categóricas
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print(f"Dataset cargado correctamente:")
print(f"- Total de filas: {len(df)}")
print(f"- Variables numéricas ({len(num_cols)}): {num_cols}")
print(f"- Variables categóricas ({len(cat_cols)}): {cat_cols}")

## 3. División del Dataset (Train / Test)

Dividimos los datos en un **80% para entrenamiento** y un **20% para evaluación final (Test)**.

* **`random_state=42`**: Fija la semilla aleatoria para garantizar la reproducibilidad exacta de los resultados.
* **`stratify=y`**: Asegura que el porcentaje de cancelaciones (37% aproximadamente) sea idéntico tanto en el conjunto de entrenamiento como en el de test, evitando sesgos por muestreo desfavorable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Conjunto de Entrenamiento (Train): {X_train.shape[0]} registros")
print(f"Conjunto de Prueba (Test): {X_test.shape[0]} registros")

## 4. Construcción del Pipeline de Preprocesamiento

Para tratar adecuadamente los datos antes de entrenar los algoritmos, definimos transformaciones específicas para cada tipo de columna:

1. **Pipeline Numérico**:
   * **`SimpleImputer(strategy="median")`**: Rellena valores ausentes con la mediana (más robusta que la media ante valores atípicos/outliers).
   * **`StandardScaler()`**: Modifica las variables para que tengan media 0 y desviación estándar 1. Vital para la Regresión Logística.

2. **Pipeline Categórico**:
   * **`SimpleImputer(strategy="most_frequent")`**: Rellena faltantes con la moda (el valor más común).
   * **`OneHotEncoder(handle_unknown="ignore")`**: Convierte texto en variables binarias (0 o 1). `handle_unknown="ignore"` evita errores si en el conjunto de prueba aparece una categoría nunca antes vista durante el entrenamiento.

In [ ]:
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols)
])

## 5. Entrenamiento y Evaluación Competitiva de Modelos

Evaluaremos dos algoritmos de naturaleza muy distinta:
1. **Regresión Logística**: Un modelo lineal paramétrico. Se usa `max_iter=1000` para asegurar que el algoritmo converja con datasets complejos.
2. **Árbol de Decisión**: Un modelo no paramétrico basado en reglas jerárquicas ("if/else").

**Buena práctica**: Colocamos el `preprocessor` y el modelo dentro de un `Pipeline` unificado. De esta forma, las transformaciones se ajustan solo con el conjunto de entrenamiento (`X_train`), eliminando cualquier riesgo de fuga de datos hacia `X_test`.

In [ ]:
# Definición de modelos a comparar
models = {
    "Regresión logística": LogisticRegression(max_iter=1000),
    "Árbol de decisión": DecisionTreeClassifier(random_state=42)
}

results = {}
tabla_filas = []

# Bucle de entrenamiento y extracción de métricas
for name, clf in models.items():
    # Pipeline completo: Preprocesado + Modelo
    pipe = Pipeline([
        ("prep", preprocessor),
        ("clf", clf)
    ])
    
    # Entrenamiento
    pipe.fit(X_train, y_train)
    
    # Predicciones
    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]
    
    # Cálculo de métricas
    f1 = f1_score(y_test, preds)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    
    # Guardar en formato legible (formato en español con coma decimal)
    tabla_filas.append([
        name,
        f"{f1:.4f}".replace(".", ","),
        f"{acc:.4f}".replace(".", ","),
        f"{prec:.4f}".replace(".", ","),
        f"{rec:.4f}".replace(".", ","),
        f"{auc:.4f}".replace(".", ",")
    ])
    
    results[name] = pipe

## 6. Visualización Integrada: Matrices de Confusión, Curvas ROC y Tabla de Métricas

In [ ]:
# Crear figura principal con GridSpec (3 filas, 2 columnas)
fig = plt.figure(figsize=(11, 9))
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.4])

# Graficar Matrices de Confusión y Curvas ROC para cada modelo
for i, (name, pipe) in enumerate(results.items()):
    ax_cm = fig.add_subplot(gs[i, 0])
    ax_roc = fig.add_subplot(gs[i, 1])
    
    # Matriz de Confusión
    ConfusionMatrixDisplay.from_estimator(pipe, X_test, y_test, ax=ax_cm, cmap="Blues")
    ax_cm.set_title(f"Matriz de Confusión - {name}", fontsize=11, fontweight="bold")
    
    # Curva ROC
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax_roc)
    ax_roc.set_title(f"Curva ROC - {name}", fontsize=11, fontweight="bold")

# Tabla de métricas en la parte inferior de la figura
ax_table = fig.add_subplot(gs[2, :])
ax_table.axis("off")

cols = ["Modelo", "F1-Score", "Accuracy", "Precisión", "Recall", "ROC-AUC"]
tabla_plot = ax_table.table(
    cellText=tabla_filas,
    colLabels=cols,
    loc="center",
    cellLoc="center"
)

tabla_plot.scale(1, 1.5)
tabla_plot.set_fontsize(10)

plt.tight_layout()
plt.show()

## 7. Interpretación de Resultados y Recomendaciones de Negocio

### Comparativa de Métricas:

| Modelo | F1-Score | Accuracy | Precisión | Recall | ROC-AUC |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Regresión logística** | 0,7321 | 0,8195 | **0,8130** | 0,6659 | **0,8962** |
| **Árbol de decisión** | **0,8085** | **0,8574** | 0,8047 | **0,8122** | 0,8503 |

---

### Conclusiones Clave:

1. **El Árbol de Decisión es el GANADOR para clasificación directa:**
   * **Mayor F1-Score (0,8085 vs 0,7321):** Muestra un equilibrio muy superior.
   * **Enorme diferencia en Recall (0,8122 vs 0,6659):** El Árbol detecta el **81,2% de todas las cancelaciones reales**, mientras que la Regresión Logística deja escapar a más de un tercio de los clientes que cancelan (solo detecta el 66,6%).

2. **La Regresión Logística destaca en la ordenación por probabilidades:**
   * **Mayor ROC-AUC (0,8962 vs 0,8503):** La regresión produce probabilidades bien calibradas y continuas. El árbol, al no estar podado, genera regiones discretas y "en codo".

---

### Recomendación Operativa:
Si el objetivo principal del hotel es **interceptar el mayor número de cancelaciones reales** para ofrecer promociones o reajustar el inventario de habitaciones, **el Árbol de Decisión es la mejor opción**. 

*Siguiente paso sugerido*: Aplicar técnicas de poda (`max_depth`, `min_samples_leaf`) al Árbol de Decisión o probar un modelo de ensamble como **Random Forest** o **XGBoost** para combinar el alto Recall del árbol con la estabilidad de probabilidad de la regresión.